# CWRU Bearing Data — Exploration (Phase 2)

**Goal:** Understand the data, visualize healthy vs fault, decide windowing strategy.

## 2.1 Load data and inspect

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Project root (works when cwd is ml/ or ml/notebooks/)
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.load_data import DATA_DIR, load_dataset, load_mat_file

In [ ]:
data = load_dataset(DATA_DIR)
print(f"Loaded {len(data)} files\n")

for name, (signal, rate, rpm) in sorted(data.items()):
    dur = len(signal) / rate
    print(f"{name:20} | {len(signal):>8,} samples | {rate:>6} Hz | {dur:.2f}s | RPM {rpm}")

In [ ]:
# Derive labels from filename
def get_label(name: str) -> str:
    if name.startswith("Normal"):
        return "normal"
    if name.startswith("IR"):
        return "inner_race"
    if name.startswith("B"):
        return "ball"
    if name.startswith("OR"):
        return "outer_race"
    return "unknown"

labels = {name: get_label(name) for name in data.keys()}
from collections import Counter
print("Label counts:", Counter(labels.values()))

## 2.2 Plot: healthy vs fault (time series)

In [ ]:
def plot_signal(name: str, signal: np.ndarray, rate: int, duration_s: float = 0.1, ax=None):
    n = int(rate * duration_s)
    t = np.arange(n) / rate
    if ax is None:
        _, ax = plt.subplots()
    ax.plot(t, signal[:n], linewidth=0.8)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
    return ax

In [ ]:
# Compare Normal vs each fault type (first 0.1s)
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
axes = axes.flatten()

normal_key = [k for k in data if k.startswith("Normal")][0]
fault_keys = [k for k in data if not k.startswith("Normal")][:3]  # IR, B, OR

for i, (key, ax) in enumerate(zip([normal_key] + fault_keys, axes)):
    sig, rate, _ = data[key]
    plot_signal(key, sig, rate, duration_s=0.1, ax=ax)

plt.tight_layout()
plt.savefig(ROOT / "notebooks" / "explore_healthy_vs_fault.png", dpi=120)
plt.show()
print("Saved: notebooks/explore_healthy_vs_fault.png")

In [ ]:
# Side-by-side: Normal vs Inner Race (same time scale)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

ir_key = [k for k in data if k.startswith("IR")][0]
sig_n, rate_n, _ = data[normal_key]
sig_f, rate_f, _ = data[ir_key]

plot_signal(f"{normal_key} (healthy)", sig_n, rate_n, 0.05, ax1)
plot_signal(f"{ir_key} (inner race fault)", sig_f, rate_f, 0.05, ax2)
plt.tight_layout()
plt.show()

## 2.3 Optional: FFT / spectrum

In [ ]:
# Frequency spectrum — fault often shows characteristic peaks
def plot_spectrum(signal: np.ndarray, rate: int, ax=None, max_freq: float = 4000):
    n = len(signal)
    fft_vals = np.fft.rfft(signal)
    fft_freq = np.fft.rfftfreq(n, 1 / rate)
    mag = np.abs(fft_vals)[: int(max_freq * n / rate)]
    freq = fft_freq[: len(mag)]
    if ax is None:
        _, ax = plt.subplots()
    ax.plot(freq, mag, linewidth=0.6)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude")
    ax.set_xlim(0, max_freq)
    ax.grid(True, alpha=0.3)
    return ax

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

ir_key = [k for k in data if k.startswith("IR")][0]
sig_n, rate_n, _ = data[normal_key]
sig_f, rate_f, _ = data[ir_key]

plot_spectrum(sig_n, rate_n, ax1, max_freq=3000)
ax1.set_title("Normal — spectrum")
plot_spectrum(sig_f, rate_f, ax2, max_freq=3000)
ax2.set_title(f"{ir_key} (inner race fault) — spectrum")
plt.tight_layout()
plt.show()

## 2.4 Windowing strategy

In [ ]:
# For ML: split each signal into fixed-length windows
WINDOW_SAMPLES = 1024  # ~0.085s @ 12kHz
STEP_SAMPLES = 512     # 50% overlap

def count_windows(signal_len: int) -> int:
    """Number of windows with given step."""
    if signal_len < WINDOW_SAMPLES:
        return 0
    return 1 + (signal_len - WINDOW_SAMPLES) // STEP_SAMPLES

print("Windowing: window=1024, step=512 (50% overlap)\n")
total = 0
for name, (sig, rate, _) in sorted(data.items()):
    n = count_windows(len(sig))
    total += n
    print(f"{name:20} | {n:>5} windows")
print(f"\nTotal windows: {total}")

In [ ]:
# Summary: ready for Phase 3 (feature engineering)
print("Phase 2 deliverable:")
print("  - Labels: normal, inner_race, ball, outer_race")
print(f"  - Window: {WINDOW_SAMPLES} samples, step {STEP_SAMPLES}")
print(f"  - Total samples for ML: ~{total} windows")